# ML5 - Decision trees

## 1. Import, Load data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from category_encoders.count import CountEncoder 
from sklearn.impute import SimpleImputer

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
%reload_ext autoreload
%autoreload 2

In [4]:
training_data = pd.read_csv('../datasets/training.csv', index_col='RefId', parse_dates=['PurchDate'])

In [5]:
training_data.head(1)

,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
RefId,,,,,,,,,,,,,,,,,,,,,
1,0,2009-12-07,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,RED,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113


In [6]:
training_data.columns

Index(['IsBadBuy', 'PurchDate', 'Auction', 'VehYear', 'VehicleAge', 'Make',
       'Model', 'Trim', 'SubModel', 'Color', 'Transmission', 'WheelTypeID',
       'WheelType', 'VehOdo', 'Nationality', 'Size', 'TopThreeAmericanName',
       'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice',
       'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice',
       'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice',
       'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice',
       'PRIMEUNIT', 'AUCGUART', 'BYRNO', 'VNZIP1', 'VNST', 'VehBCost',
       'IsOnlineSale', 'WarrantyCost'],
      dtype='object')

## Train/valid/test split

- drop `VehYear` (collinear with `VehicleAge`), `WheelTypeID` (collinear with `WheelType`), `SubModel`

In [7]:
training_data = training_data.drop(['VehYear', 'WheelTypeID', 'SubModel'], axis=1)

- sort by `PurchDate` and split into 1/3 parts

In [8]:
size = len(training_data) // 3 + 1
sorted_by_purchdate = training_data.sort_values(by='PurchDate')
train = sorted_by_purchdate.iloc[:size]
valid = sorted_by_purchdate.iloc[size:size*2]
test = sorted_by_purchdate.iloc[size*2:]

In [9]:
train.PurchDate.iloc[0] < valid.PurchDate.iloc[0] < test.PurchDate.iloc[0]

True

In [10]:
train = train.drop(['PurchDate'], axis=1)
valid = valid.drop(['PurchDate'], axis=1)
test = test.drop(['PurchDate'], axis=1)

- split into X and y

In [11]:
X_train = train.drop(['IsBadBuy'], axis=1)
y_train = train['IsBadBuy']

X_valid = valid.drop(['IsBadBuy'], axis=1)
y_valid = valid['IsBadBuy']

X_test = test.drop(['IsBadBuy'], axis=1)
y_test = test['IsBadBuy']

## Features preprocessing

In [12]:
imputer = SimpleImputer(strategy='mean')

In [13]:
count_enc = CountEncoder()

### count encoding

In [14]:
X_train = count_enc.fit_transform(X_train)
X_valid = count_enc.transform(X_valid)
X_test = count_enc.transform(X_test)

### NaN imputation

In [15]:
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_valid = pd.DataFrame(imputer.transform(X_valid), columns=X_valid.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# 2. MyDecisionTree

## classifier

split criterion (impurity $H$): \
$$\large H = Gini=1 − ∑_{i=1}^n p_i^2$$ \
$p_i$ is the probability (share) of class $i$ in the node
- lower values indicate cleaner and more homogeneous nodes
- nodes become pure when all samples belong to one class

In [16]:
from dtree_fast_optim import MyDecisionTreeClassifier

In [17]:
tree_clf = MyDecisionTreeClassifier(max_depth=30)

In [18]:
%%time
tree_clf.fit(X_train, y_train)

CPU times: user 9.32 s, sys: 31.3 ms, total: 9.35 s
Wall time: 9.35 s


In [19]:
tree_clf.predict_proba(X_valid)

array([[1., 0.],
       [1., 0.],
       [1., 0.],
       ...,
       [1., 0.],
       [0., 1.],
       [1., 0.]], shape=(24328, 2))

In [20]:
tree_clf.predict(X_valid)

array([0, 0, 0, ..., 0, 1, 0], shape=(24328,))

In [21]:
tree_clf.print_tree()

[WheelType > 5773.500] (samples=24328)
    R-> [VehicleAge > 5.500] (samples=23168)
        R-> [VehBCost > 4197.500] (samples=4703)
            R-> [MMRCurrentAuctionAveragePrice > 4598.500] (samples=3229)
                R-> [Make > 7.000] (samples=1420)
                    R-> [MMRAcquisitionAuctionCleanPrice > 5363.500] (samples=1417)
                        R-> [MMRAcquisitionAuctionCleanPrice > 5387.500] (samples=1244)
                            R-> [VehBCost > 8282.500] (samples=1239)
                                R-> [VehOdo > 99197.000] (samples=463)
                                    R-> Leaf: probas=(np.float64(0.0), np.float64(1.0)) (samples=1)
                                    L-> [MMRAcquisitionAuctionCleanPrice > 5646.500] (samples=462)
                                        R-> [VNZIP1 > 13972.500] (samples=461)
                                            R-> [VehBCost > 11615.000] (samples=460)
                                                R-> [Auction > 9665.

In [22]:
tree_clf.tree.right_leaf.right_leaf.right_leaf.right_leaf.right_leaf.probas

## regressor

split criterion: \
$$\large H = SSE = ∑_{i=1}^n (y_i - \bar y)^2$$ \
$SSE$ - sum of squared error
- $SSE$ criterion produces exactly the same split as $MSE$ would, since denominator term in $MSE$ is canceled out by taking weighted average of impurities in leaves:
$$Branch(X_m,j,t) = \  ∣X_m∣*H(X_m) \ − \ ∣X_l∣*H(X_l) \ − \ ∣X_r∣*H(X_r)$$
- lower values indicate nodes with lower variance
- nodes become pure when all samples have the same value (zero variance)

In [23]:
from my_GBDT import MyDecisionTreeRegressor

In [24]:
regression_dataset = pd.read_json('../datasets/regression-train.json')
regression_dataset.dropna(inplace=True)
regression_dataset = regression_dataset.reset_index()

X_train_regression = regression_dataset.loc[:20000, ['bathrooms', 'bedrooms', 'latitude', 'longitude']]
y_train_regression = regression_dataset.loc[:20000, 'price']
X_test_regression = regression_dataset.loc[20000:30000, ['bathrooms', 'bedrooms', 'latitude', 'longitude']]
y_test_regression = regression_dataset.loc[20000:30000, 'price']

In [25]:
gb_reg = MyDecisionTreeRegressor(max_depth=3)
gb_reg.fit(X_train_regression, y_train_regression)
gb_reg.predict(X_test_regression)

array([3117.28408564, 3117.28408564, 3117.28408564, ..., 3117.28408564,
       3117.28408564, 5362.9336801 ], shape=(10001,))

## ExtraRandomTree

- split values are chosen randomly for a given feature ->
    - faster fit
    - lower variance (less prone to overfitting)

In [26]:
from dtree_fast_optim import MyExtraRandomizedTreeClassifier

In [27]:
ert_clf = MyExtraRandomizedTreeClassifier(max_depth=50, random_state=21)

In [28]:
%%time
ert_clf.fit(X_train, y_train)

CPU times: user 1.39 s, sys: 11.4 ms, total: 1.4 s
Wall time: 1.4 s


In [29]:
ert_clf.predict_proba(X_valid)

array([[1., 0.],
       [0., 1.],
       [1., 0.],
       ...,
       [1., 0.],
       [0., 1.],
       [1., 0.]], shape=(24328, 2))

In [30]:
ert_clf.predict(X_valid)

array([0, 1, 0, ..., 0, 1, 0], shape=(24328,))

In [31]:
ert_clf.print_tree()

[MMRCurrentAuctionCleanPrice > 4996.000] (samples=24328)
    R-> [MMRCurrentAuctionAveragePrice > 5059.000] (samples=18602)
        R-> [VehicleAge > 5.000] (samples=14481)
            R-> [MMRCurrentRetailCleanPrice > 8706.000] (samples=1228)
                R-> [Auction > 4998.000] (samples=626)
                    R-> [MMRCurrentRetailAveragePrice > 7856.000] (samples=475)
                        R-> [Make > 9.000] (samples=389)
                            R-> [MMRAcquisitionAuctionCleanPrice > 7803.000] (samples=380)
                                R-> [TopThreeAmericanName > 4629.000] (samples=276)
                                    R-> [MMRAcquisitionAuctionCleanPrice > 10412.000] (samples=133)
                                        R-> [Auction > 5230.000] (samples=22)
                                            R-> Leaf: probas=(np.float64(1.0), np.float64(0.0)) (samples=18)
                                            L-> [MMRAcquisitionAuctionCleanPrice > 10612.000] (samples

# 3, 4. MyDecisionTree Gini score

In [32]:
from sklearn.metrics import roc_auc_score

In [33]:
def my_gini(y_true, y_score):
    return 2 * roc_auc_score(y_true=y_true, y_score=y_score) - 1

In [34]:
# from my_classification import my_gini
from sklearn.tree import DecisionTreeClassifier

In [35]:
sklearn_clf = DecisionTreeClassifier()
sklearn_clf.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curre

In [36]:
my_tree_clf_reg = MyDecisionTreeClassifier(max_depth=5).fit(X_train, y_train)

In [37]:
print(f"MyDecisionTreeClassifier depth=5 gini score: {my_gini(y_score=my_tree_clf_reg.predict_proba(X_valid)[:, 1], y_true=y_valid)}")
print(f"MyDecisionTreeClassifier depth=30 gini score: {my_gini(y_score=tree_clf.predict_proba(X_valid)[:, 1], y_true=y_valid)}")
print(f"sklearn DTree gini score: {my_gini(y_score=sklearn_clf.predict_proba(X_valid)[:, 1], y_true=y_valid)}")
print(f"MyExtraRandomizedTree gini score: {my_gini(y_score=ert_clf.predict_proba(X_valid)[:, 1], y_true=y_valid)}")

MyDecisionTreeClassifier depth=5 gini score: 0.45755734638477996
MyDecisionTreeClassifier depth=30 gini score: 0.19929711522631877
sklearn DTree gini score: 0.21287627703556922
MyExtraRandomizedTree gini score: 0.09370496759877267


- sklearn `DecisionTreeClassifier` with default parameters has no regularization
- while one instance of `MyDecisionTreeClassifier` is regularized a bit by setting `max_depth=5`
- so sklearn classifier is **clearly overfitted**

# 5. MyRandomForestClassifier

- idea: reduce variance by aggregating predictions of many uncorrelated strong learners
- techniques used to decrease variance between trees in the forest:
    - **bootstrap aggregating** (bagging): every tree is fitted on a random subset of training samples (taken with replacement)
    - **feature subspace method** (feature bagging): every node is fitted using only a random subset of features (size of the subset can be set with `max_features` parameter)
    - `max_depth` of a tree is deliberately set to be high so that a tree "overfits" to capture its "view" of the training data
- trees are fitted independently, so it is easy to add **multiprocessing** both for `fit` and `predict` methods

In [38]:
from dtree_fast_optim import MyRandomForestClassifier

In [39]:
rf = MyRandomForestClassifier(max_depth=40, max_features='sqrt', number_of_trees=50, random_state=42)

In [40]:
rf.fit(X_train, y_train)

Training 50 trees with n_jobs=-1:


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   19.6s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   26.4s finished


### `joblib` multiprocessing vs. single-cpu implementation

In [41]:
n_jobs=-1
MyRandomForestClassifier(max_depth=30, max_features='sqrt', number_of_trees=16, random_state=21, n_jobs=n_jobs).fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


Training 16 trees with n_jobs=-1:


[Parallel(n_jobs=-1)]: Done  16 out of  16 | elapsed:    9.3s finished


In [42]:
n_jobs=1
MyRandomForestClassifier(max_depth=30, max_features='sqrt', number_of_trees=16, random_state=21, n_jobs=n_jobs).fit(X_train, y_train)

Training 16 trees with n_jobs=1:


[Parallel(n_jobs=1)]: Done  16 out of  16 | elapsed:   27.8s finished


### gini score

In [43]:
rf.predict_proba(X_valid)

array([[0.9 , 0.1 ],
       [0.96, 0.04],
       [0.88, 0.12],
       ...,
       [0.84, 0.16],
       [0.68, 0.32],
       [0.83, 0.17]], shape=(24328, 2))

In [44]:
rf.predict(X_valid)

array([0, 0, 0, ..., 0, 0, 0], shape=(24328,))

In [45]:
rf.trees[6].print_tree()

[VehicleAge > 5.500] (samples=24328)
    R-> [WheelType > 5773.500] (samples=5122)
        R-> [MMRCurrentRetailCleanPrice > 5733.000] (samples=4801)
            R-> [Size > 305.000] (samples=2222)
                R-> [Trim > 5.500] (samples=2127)
                    R-> [MMRAcquisitonRetailCleanPrice > 8207.500] (samples=2097)
                        R-> [VehBCost > 11615.000] (samples=620)
                            R-> [VehOdo > 75114.500] (samples=4)
                                R-> Leaf: probas=(np.float64(1.0), np.float64(0.0)) (samples=1)
                                L-> Leaf: probas=(np.float64(0.0), np.float64(1.0)) (samples=3)
                            L-> [VehOdo > 99035.500] (samples=616)
                                R-> Leaf: probas=(np.float64(0.0), np.float64(1.0)) (samples=1)
                                L-> [Make > 7.000] (samples=615)
                                    R-> [MMRAcquisitionAuctionCleanPrice > 9300.500] (samples=613)
                     

In [46]:
print(f"MyRandomForestClassifier gini score: {my_gini(y_score=rf.predict_proba(X_valid)[:, 1], y_true=y_valid)}")

MyRandomForestClassifier gini score: 0.39744365297462636


In [47]:
from sklearn.ensemble import RandomForestClassifier

In [48]:
sk_rf = RandomForestClassifier().fit(X_train, y_train)

In [49]:
print(f"sklearn RandomForest gini score: {my_gini(y_score=sk_rf.predict_proba(X_valid)[:, 1], y_true=y_valid)}")

sklearn RandomForest gini score: 0.42261572737236475


# 6. My Gradient Boosting Decision Trees Classifier

- boosting idea: iteratively fit models so that each successive model learns to predict and correct the mistakes made by all previous models
- gradient boosting: consider contribution of a model on the next iteration as a delta in Loss function
- let's take binary cross-entropy:
$$ L(y, p) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right] $$
$p_i$ - predicted class 1 probability\
$y$ - true class (1/0)
- now substitute with $p_i = \sigma(z_i)$:
$$ L(y, z) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\sigma(z_i)) + (1 - y_i) \log(1 - \sigma(z_i)) \right]$$
- here, instead of calculating logits using linear model and taking derivative of L by weights, we take derivative of L only by logits:
$$\frac{\partial L}{\partial z} = \sigma(z) - y = p(z) - y$$ 

- we will use this derivative to achieve first-order Taylor approximation of our L:
$$L(y, z + \Delta z) \approx L(y, z) + \frac{\partial L}{\partial z} \Delta z \rightarrow min_{\Delta z}$$
- and this minimization taks is solved simply by setting:
$$ \Delta z = - \frac{\partial L}{\partial z} = y - \sigma(z)$$
- HERE GB comes into play: we decide that out algorithm produces logits:
$$z_k = ∑_{i=0}^{k}a_k(x)$$
$$p_k(x) = \sigma (z_𝑘(x))$$

- and incrementing $z$ is just like adding the successive model correcitve:
$$z + \Delta z = a_k(x) + a_{k+1}(x)$$
$$\Delta z = a_{k+1}(x) $$
- summing all up, we get the rule to fit every successive **regressor** for this task:
$$ a_{k+1}(x) \approx - \frac{\partial L}{\partial z} = y - \sigma(z) = y - p(z)$$
    - which is very intuitive: the antigradient will force $a_{k+1}(x)$ to decrease logits for y=0 and increase them for y=1
- after that the ensemble is updated:
$$ z_{k+1}(x) ← z_k(x) + a_{k+1}(x) $$
- derivative of Loss by logits is used for gradient boosting classification, because logits suit better for regressor predictions (since logits $\in \mathbb{R}$, and probabilities $\in [0, 1]$)


In [50]:
from my_GBDT import MyGBDTClassifier

In [51]:
my_gbdt = MyGBDTClassifier(max_depth=3, max_features='sqrt', number_of_trees=100, learning_rate=0.1, random_state=21)

In [52]:
my_gbdt.fit(X_train, y_train)

In [53]:
my_gbdt.predict_proba(X_valid)

array([[0.84953797, 0.15046203],
       [0.86278424, 0.13721576],
       [0.86533571, 0.13466429],
       ...,
       [0.86101986, 0.13898014],
       [0.83309485, 0.16690515],
       [0.85745181, 0.14254819]], shape=(24328, 2))

In [54]:
print(f"My GBDT gini score: {my_gini(y_score=my_gbdt.predict_proba(X_valid)[:, 1], y_true=y_valid)}")

My GBDT gini score: 0.47079475176820296


# 7. CatBoost vs LightGBM vs XGBoost

1) Feature processing
- CatBoost and LightGDM treat categorical features specifically (e.g. using one-hot for low cardinality features and ordered target statistics encoding for high cardinality features), so one-hot encoding is not needed on preprocessing
- CatBoost supports text data

2) Tree structure
- CatBoost uses the same predicate for a level (oblivious trees) -> symmetric trees (prevents  overfit)
- LightGBM grows the tree depth-wise: every time the node with the highest gain is split (its depth not taken into account), improves speed by not building the whole tree

3) Splitting
- LightGDM uses GOSS (Gradient-based One-Side Sampling) - taking all samples with high gradient and only a random part of those with small gradient to build tree, improves speed
- all libraries have histogram-based split for speed improvement (numerical features are binned)
- CatBoost supports Ordered Boosting to decrease overfitting (antigradient is predicted using a model fitted on other samples using permutations)

4) Task type
- apart from regression and classification, CatBoost has sophisticated tools for ranking tasks: YetiRank, PairLogit etc.

5) Overall Speed
- for CPU training, LightGBM is the fastest
- for GPU training and prediction time, CatBoost is the fastest

## Hyperparameters optimization

In [55]:
import optuna
from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb

## MyGBDT

In [56]:
def my_objective(trial):

    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 5),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.5, log=True
        ),
        "number_of_trees": trial.suggest_int("number_of_trees", 20, 60),
        "max_features": trial.suggest_float("max_features", 0.05, 1)
    }

    model = MyGBDTClassifier(**params)

    model.fit(
        X_train, y_train
    )

    preds = model.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, preds)

    gini = 2 * auc - 1
    return gini

In [57]:
# myGBDT_study = optuna.create_study(direction="maximize", study_name="myGBDT__gini")
# myGBDT_study.optimize(my_objective, n_trials=20)

## CatBoost

In [58]:
def catboost_objective(trial):

    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",          
        "iterations": trial.suggest_int("iterations", 1000, 10000),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.5, log=True
        ),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1e-3, 10.0, log=True
        ),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature", 0.0, 1.0
        ),
        "random_strength": trial.suggest_float(
            "random_strength", 0.0, 2.0
        ),
        "verbose": False,
        "random_seed": 42,

        "od_type": "Iter",
        "od_wait": trial.suggest_int("od_wait", 50, 200),
    }

    model = CatBoostClassifier(**params)

    model.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=100,
        use_best_model=True
    )

    preds = model.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, preds)

    gini = 2 * auc - 1
    return gini

In [59]:
cb_study = optuna.create_study(direction="maximize", study_name="catboost_gini")
cb_study.optimize(catboost_objective, n_trials=50)

[I 2026-02-03 12:06:44,698] A new study created in memory with name: catboost_gini
[I 2026-02-03 12:06:45,342] Trial 0 finished with value: 0.48016592651548384 and parameters: {'iterations': 4945, 'learning_rate': 0.08829766263678075, 'depth': 6, 'l2_leaf_reg': 0.07567657908301, 'bagging_temperature': 0.13322003820417672, 'random_strength': 0.5691681900908725, 'od_wait': 137}. Best is trial 0 with value: 0.48016592651548384.
[I 2026-02-03 12:06:45,984] Trial 1 finished with value: 0.47632435132435136 and parameters: {'iterations': 6733, 'learning_rate': 0.09363416669376208, 'depth': 6, 'l2_leaf_reg': 0.02249288930852759, 'bagging_temperature': 0.7977225295668283, 'random_strength': 0.9246830605854239, 'od_wait': 134}. Best is trial 0 with value: 0.48016592651548384.
[I 2026-02-03 12:06:47,029] Trial 2 finished with value: 0.4613355431939503 and parameters: {'iterations': 7088, 'learning_rate': 0.36646134351415716, 'depth': 9, 'l2_leaf_reg': 8.720640890045232, 'bagging_temperature': 0.8

## LightGBM

In [60]:
def lgb_objective(trial):

    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "verbosity": -1,

        # trees
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),

        # regularization
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 300),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 10.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 10.0),

        # sampling
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),

        # learning
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 5000)
    }

    model = lgb.LGBMClassifier(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )

    preds = model.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, preds)

    gini = 2 * auc - 1
    return gini


In [61]:
lgb_study = optuna.create_study(direction="maximize", study_name="lightgbm_gini")
lgb_study.optimize(lgb_objective, n_trials=50)

[I 2026-02-03 12:09:10,706] A new study created in memory with name: lightgbm_gini
[I 2026-02-03 12:09:15,554] Trial 0 finished with value: 0.47520026612946964 and parameters: {'num_leaves': 184, 'min_data_in_leaf': 72, 'lambda_l1': 1.141308841833072, 'lambda_l2': 8.329295158241008, 'feature_fraction': 0.9982960310621077, 'bagging_fraction': 0.7862790422425036, 'bagging_freq': 8, 'learning_rate': 0.03786813472840579, 'n_estimators': 2759}. Best is trial 0 with value: 0.47520026612946964.
[I 2026-02-03 12:09:19,088] Trial 1 finished with value: 0.47229900271935676 and parameters: {'num_leaves': 238, 'min_data_in_leaf': 85, 'lambda_l1': 1.106781805467364, 'lambda_l2': 3.764745201849756, 'feature_fraction': 0.7950243860455661, 'bagging_fraction': 0.7368923782316088, 'bagging_freq': 10, 'learning_rate': 0.056842044130702735, 'n_estimators': 3769}. Best is trial 0 with value: 0.47520026612946964.
[I 2026-02-03 12:09:20,219] Trial 2 finished with value: 0.4783283267796543 and parameters: {'n

## XGBoost

In [62]:
def xgb_objective(trial):

    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "verbosity": 0,

        # trees
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 100.0),
        "n_estimators": trial.suggest_int("n_estimators", 100, 2000),
        "max_bin": trial.suggest_int("max_bin", 64, 512), 

        # regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 10.0),

        # sampling
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        # learning
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
    }

    model = xgb.XGBClassifier(
        **params,
        use_label_encoder=False,
        early_stopping_rounds=100
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    preds = model.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, preds)

    gini = 2 * auc - 1
    return gini


In [63]:
xgb_study = optuna.create_study(direction="maximize",  study_name="xgboost_gini")
xgb_study.optimize(xgb_objective, n_trials=50)

[I 2026-02-03 12:10:56,764] A new study created in memory with name: xgboost_gini
[I 2026-02-03 12:10:58,118] Trial 0 finished with value: 0.48175189436251364 and parameters: {'max_depth': 9, 'min_child_weight': 41.1206954892313, 'n_estimators': 1429, 'max_bin': 320, 'reg_alpha': 0.3052504012864432, 'reg_lambda': 2.7460585944711635, 'subsample': 0.7209034153768087, 'colsample_bytree': 0.6619493102846014, 'learning_rate': 0.02211396344802843}. Best is trial 0 with value: 0.48175189436251364.
[I 2026-02-03 12:11:00,758] Trial 1 finished with value: 0.4830310505089268 and parameters: {'max_depth': 6, 'min_child_weight': 21.053340032131082, 'n_estimators': 616, 'max_bin': 511, 'reg_alpha': 5.181385431846136, 'reg_lambda': 8.767093945916479, 'subsample': 0.6224134674615179, 'colsample_bytree': 0.652847712675016, 'learning_rate': 0.010846624296797397}. Best is trial 1 with value: 0.4830310505089268.
[I 2026-02-03 12:11:02,618] Trial 2 finished with value: 0.48567030314817927 and parameters: 

In [64]:
# print("MyGBDT")
# print("Best Gini:", myGBDT_study.best_value)
# print("Best params:")
# for k, v in myGBDT_study.best_params.items():
#     print(f"{k}: {v}")
# print("=" * 15)
print("CatBoost")
print("Best Gini:", cb_study.best_value)
print("Best params:")
for k, v in cb_study.best_params.items():
    print(f"{k}: {v}")
print("=" * 15)
print("LightGBM")
print("Best Gini:", lgb_study.best_value)
print("Best params:")
for k, v in lgb_study.best_params.items():
    print(f"{k}: {v}")
print("=" * 15)
print("XGBoost")
print("Best Gini:", xgb_study.best_value)
print("Best params:")
for k, v in xgb_study.best_params.items():
    print(f"{k}: {v}")

CatBoost
Best Gini: 0.49395265103229713
Best params:
iterations: 9433
learning_rate: 0.00837253258624514
depth: 6
l2_leaf_reg: 5.234393448708352
bagging_temperature: 0.33354919553537604
random_strength: 0.8291672567083168
od_wait: 93
LightGBM
Best Gini: 0.4941162341604819
Best params:
num_leaves: 17
min_data_in_leaf: 25
lambda_l1: 6.48991344601562
lambda_l2: 1.888605489526504
feature_fraction: 0.8194633159766777
bagging_fraction: 0.8837339264443308
bagging_freq: 5
learning_rate: 0.03199432031140333
n_estimators: 2897
XGBoost
Best Gini: 0.48909370646096306
Best params:
max_depth: 5
min_child_weight: 5.076287217432484
n_estimators: 1503
max_bin: 486
reg_alpha: 8.205783382495923
reg_lambda: 2.1684190188620227
subsample: 0.7413830520387207
colsample_bytree: 0.9092148582782597
learning_rate: 0.015635527830789378


In [65]:
best_clf = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",  
    **cb_study.best_params
)

In [66]:
best_clf.fit(X_train, y_train, 
             eval_set=(X_valid, y_valid),
             early_stopping_rounds=100,
             verbose=1000)

0:	test: 0.6916566	best: 0.6916566 (0)	total: 5.73ms	remaining: 54s
1000:	test: 0.7469692	best: 0.7470680 (921)	total: 4.85s	remaining: 40.9s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7470680181
bestIteration = 921

Shrink model to first 922 iterations.


In [67]:
print(f"best model train gini score: {my_gini(y_score=best_clf.predict_proba(X_train)[:, 1], y_true=y_train)}")
print(f"best model valid gini score: {my_gini(y_score=best_clf.predict_proba(X_valid)[:, 1], y_true=y_valid)}")
print(f"best model test gini score: {my_gini(y_score=best_clf.predict_proba(X_test)[:, 1], y_true=y_test)}")

best model train gini score: 0.5894508632463034
best model valid gini score: 0.4941360361714344
best model test gini score: 0.47313919698277873


- the model result is very stable between train, valid and test -> no overfitting!

## 9. MyExtraTreesClassifier

- same idea as in `RandomForest` 
- instead of bootstrap sampling, ExtraRandomTree random split value selection is used to decrease variance and correlation between trees (though sample bagging is still an option in some implementations)

In [68]:
from dtree_fast_optim import MyExtraTreesClassifier

In [69]:
ex_trees = MyExtraTreesClassifier(max_depth=40, max_features='sqrt', number_of_trees=50, random_state=42)

In [70]:
ex_trees.fit(X_train, y_train)

Training 50 trees with n_jobs=-1:


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   20.8s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   28.1s finished


In [71]:
ex_trees.predict_proba(X_valid)

array([[0.9311028 , 0.0688972 ],
       [1.        , 0.        ],
       [0.936     , 0.064     ],
       ...,
       [0.91810369, 0.08189631],
       [0.76077521, 0.23922479],
       [0.9111028 , 0.0888972 ]], shape=(24328, 2))

In [72]:
ex_trees.predict(X_valid)

array([0, 0, 0, ..., 0, 0, 0], shape=(24328,))

In [73]:
print(f"MyExtraTreesClassifier gini score: {my_gini(y_score=ex_trees.predict_proba(X_valid)[:, 1], y_true=y_valid)}")

MyExtraTreesClassifier gini score: 0.33969052055335247
